# Build and deliver the monthly channel report

*Data Analysis in Python · Lesson 4 final exercise*

Marketing leadership wants the store's monthly channel report for 2024 and 2025, as an Excel file they can open without you: a sheet per year, a row per month, and each channel's sessions, revenue and conversion rate, with the store's total.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

Add your cells between the clean sessions cell and the **Check your work** cell.

1. Add a `channel_group` column to `sessions` with your `channel_group()` from Foundations topic 12.3., and a `month` column, such as `"2024-11"`. Check that every channel group is in `CUSTOMER_CHANNELS`.
2. Create `monthly`: one row per month and channel group, and one per month with the channel group `"Total"` for all channels together, with `sessions`, `transactions`, `revenue` and `conversion_rate`.
3. Create `overall_rate`: the conversion rate of both years together.
4. Create `report`: months as rows, and for each channel group and the total, a column of sessions, one of revenue and one of conversion rate, named like `"Email sessions"`, `"Email revenue"` and `"Email conversion rate"`.
5. Write `REPORT_FILE` with a sheet per year, named `2024` and `2025`, each with that year's twelve months of `report`:
   - sessions as `#,##0`, revenue as `$#,##0.00` and conversion rates as `0.00%`
   - every column at least 16 wide
6. Read the file back, and check it against `report`.
7. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output. Then download the file from the Files panel and open it.

**Expected output**: `sessions` has a `channel_group` column with nine groups, all in `CUSTOMER_CHANNELS`, and a `month` column. `monthly` has 240 rows: 24 months of nine channel groups and a `Total`. `overall_rate` is about `0.02598` (2.60%). `report` has 24 months and 30 columns, from `Direct sessions` to `Total conversion rate`. `channel_report_2024_2025.xlsx` has the sheets **2024** and **2025**, twelve months each, with sessions as `#,##0`, revenue as `$#,##0.00` and conversion rates as `0.00%`, and every column at least 16 wide. November 2024's Email conversion rate is 7.71%, and its total 3.21%.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = [f"{year}-{month:02d}" for year in (2024, 2025) for month in range(1, 13)]
START, END = "2024-01-01", "2025-12-31"
REPORT_FILE = "channel_report_2024_2025.xlsx"
# The CRM's first-touch channels, from customers.csv (Foundations topic 12.3)
CUSTOMER_CHANNELS = {"Organic Search", "Paid Search", "Direct", "Email", "Paid Social", "Paid Video", "Organic Social", "Referral", "Display"}

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables, the cells that have run and the file you wrote. It doesn't change anything.
def check_your_work():
    import os
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    # Foundations topic 12.3's rules, to build the expected report from the clean table
    social_sites = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}
    rules = {"email": "Email", "cpc": "Paid Search", "display": "Display", "paid_social": "Paid Social", "video": "Paid Video", "organic": "Organic Search"}

    def expected_group(source, medium):
        source, medium = str(source).strip().lower(), str(medium).strip().lower()
        if source == "(direct)" and medium == "(none)":
            return "Direct"
        if medium in rules:
            return rules[medium]
        if medium == "referral":
            return "Organic Social" if source in social_sites else "Referral"
        return "Unassigned"

    sessions = found.get("sessions")
    expected = None
    if not isinstance(sessions, pd.DataFrame) or len(sessions) != 43035 or sessions["sessions"].sum() != 2506005:
        results.append((False, "sessions should be the clean table for 2024 and 2025, 43,035 rows and 2,506,005 sessions. Run the cell that builds it, and don't filter it."))
    elif "channel_group" not in sessions.columns or "month" not in sessions.columns:
        results.append((False, "sessions needs a channel_group column, from your Foundations function, and a month column such as \"2024-11\"."))
    else:
        groups = [expected_group(source, medium) for source, medium in zip(sessions["session_source"], sessions["session_medium"])]
        table = sessions[["date", "sessions", "transactions", "revenue"]].assign(channel_group=groups, month=sessions["date"].dt.strftime("%Y-%m"))
        by_group = table.groupby(["month", "channel_group"])[["sessions", "transactions", "revenue"]].sum()
        totals = table.groupby("month")[["sessions", "transactions", "revenue"]].sum()
        days = table.groupby(["month", "channel_group", "date"])[["sessions", "transactions"]].sum()
        daily_mean = (days["transactions"] / days["sessions"]).groupby(level=["month", "channel_group"]).mean()
        expected = {"by_group": by_group, "totals": totals, "daily_mean": daily_mean}
        found_groups = set(sessions["channel_group"].unique())
        if "Unassigned" in found_groups and "Display" not in found_groups:
            results.append((False, f"sessions has {(sessions['channel_group'] == 'Unassigned').sum():,} Unassigned rows, and no Display or Paid Video: that's topic 5.3's first channel_group(). Paste the one from topic 12.3."))
        elif "Organic Social" not in found_groups:
            results.append((False, "sessions has no Organic Social: your row function gives channel_group() the standardized source column. SOCIAL_SITES lists the export's spellings: use session_source."))
        elif list(sessions["channel_group"]) != groups:
            results.append((False, f"sessions' channel_group doesn't follow topic 12.3's rules on every row: it has {sorted(found_groups)}."))
        elif found_groups - CUSTOMER_CHANNELS:
            results.append((False, f"Some channel groups aren't in CUSTOMER_CHANNELS: {found_groups - CUSTOMER_CHANNELS}."))
        elif history is not None and not any("CUSTOMER_CHANNELS" in source for source in history if "CUSTOMER_CHANNELS = " not in source):
            results.append((False, "Your groups are right, but nothing in your notebook checks them against CUSTOMER_CHANNELS. Add a check, such as assert set(sessions[\"channel_group\"].unique()) <= CUSTOMER_CHANNELS."))
        else:
            results.append((True, "sessions has the nine channel groups from topic 12.3's rules, all of them names the CRM uses, and a month column."))

    monthly = found.get("monthly")
    if expected is not None:
        if not isinstance(monthly, pd.DataFrame) or not {"month", "channel_group", "sessions", "conversion_rate"} <= set(monthly.columns):
            results.append((False, "monthly should be a DataFrame with the columns month, channel_group, sessions, transactions, revenue and conversion_rate, one row per month and channel group."))
        elif "Total" not in set(monthly["channel_group"]):
            results.append((False, f"monthly has {len(monthly):,} rows and no Total. Group by month alone for the totals, label them \"Total\", and stack them with pd.concat."))
        elif len(monthly) != 240:
            results.append((False, f"monthly has {len(monthly):,} rows, {monthly['channel_group'].nunique()} channel groups with the Total, but 24 months of nine channel groups and a Total is 240."))
        else:
            results.append((True, "monthly has 240 rows: 24 months of nine channel groups and the Total."))

    overall = found.get("overall_rate")
    if isinstance(overall, (pd.DataFrame, pd.Series)):
        results.append((False, f"overall_rate is a {type(overall).__name__}, not a number. Divide two totals: sessions[\"transactions\"].sum() / sessions[\"sessions\"].sum()."))
    elif not pd.api.types.is_number(overall):
        results.append((False, f"overall_rate should be the two years' transactions over their sessions. It is {overall!r}."))
    elif abs(overall - 0.024765) < 0.000001:
        results.append((False, "overall_rate is 2.48%, the mean of 731 daily rates. The two years' rate is their total transactions over their total sessions: 2.60%."))
    elif abs(overall - 0.018344) < 0.000001:
        results.append((False, "overall_rate is 1.83%, the mean of the rows' rates, without the refund rows' NaN. Divide the totals."))
    elif abs(overall - 0.025980) > 0.000001:
        results.append((False, f"overall_rate should be sessions[\"transactions\"].sum() / sessions[\"sessions\"].sum(), about 0.02598. It is {overall!r}."))
    else:
        results.append((True, f"Over both years, {overall:.2%} of sessions converted."))

    report_file = found.get("REPORT_FILE")
    written = history is None or any("ExcelWriter" in source or "to_excel" in source for source in history)
    if expected is None:
        pass
    elif not isinstance(report_file, str) or not os.path.exists(report_file) or not written:
        results.append((False, f"{report_file} hasn't been written in this session. Write it with pd.ExcelWriter(REPORT_FILE, engine=\"openpyxl\"), a sheet per year."))
    else:
        import openpyxl
        back = pd.read_excel(report_file, sheet_name=None, index_col=0)
        book = openpyxl.load_workbook(report_file)
        names = ["Direct", "Display", "Email", "Organic Search", "Organic Social", "Paid Search", "Paid Social", "Paid Video", "Referral", "Total"]
        columns = [f"{name} {metric}" for name in names for metric in ["sessions", "revenue", "conversion rate"]]
        if sorted(back) != ["2024", "2025"]:
            results.append((False, f"{report_file} should have the sheets 2024 and 2025. It has {list(back)}."))
        else:
            problems = []
            for year, sheet in back.items():
                if list(sheet.index) != [f"{year}-{month:02d}" for month in range(1, 13)]:
                    problems.append(f"The {year} sheet should have {year}'s twelve months as rows, from {year}-01 to {year}-12, with the months as its first column. It has {list(sheet.index)[:3]}...")
                    continue
                if "Unassigned sessions" in sheet.columns:
                    problems.append(f"The {year} sheet has Unassigned columns: the channel groups come from topic 5.3's function.")
                    continue
                if missing := [column for column in columns if column not in sheet.columns]:
                    problems.append(f"The {year} sheet has no {', '.join(missing[:3])}{'...' if len(missing) > 3 else ''} column. Name them like \"Email sessions\", \"Email revenue\" and \"Email conversion rate\", with a Total for each.")
                    continue
                for name in names:
                    rows = expected["totals"].loc[sheet.index] if name == "Total" else expected["by_group"].xs(name, level="channel_group").loc[sheet.index]
                    if (sheet[f"{name} sessions"] - rows["sessions"]).abs().max() > 0.5 or (sheet[f"{name} revenue"] - rows["revenue"]).abs().max() > 0.005:
                        problems.append(f"The {year} sheet's {name} sessions or revenue don't match the month's totals.")
                        break
                    rate = sheet[f"{name} conversion rate"]
                    right = rows["transactions"] / rows["sessions"]
                    if (rate - right).abs().max() < 1e-9:
                        continue
                    if (rate - right * 100).abs().max() < 1e-7 or rate.max() > 1:
                        problems.append(f"The {year} sheet's {name} conversion rate is multiplied by 100: {rate.iloc[0]:.2f}. Keep rates as fractions, and let the 0.00% format show them.")
                    elif name == "Total" and (rate - expected["by_group"]["transactions"].div(expected["by_group"]["sessions"]).groupby(level="month").mean().loc[sheet.index]).abs().max() < 1e-9:
                        problems.append(f"The {year} sheet's Total conversion rate is the mean of the nine channels' rates. The total is all channels' transactions over all their sessions.")
                    elif name != "Total" and (rate - expected["daily_mean"].xs(name, level="channel_group").loc[sheet.index]).abs().max() < 1e-9:
                        problems.append(f"The {year} sheet's {name} conversion rate is the mean of its daily rates. A month's rate is its transactions over its sessions.")
                    else:
                        problems.append(f"The {year} sheet's {name} conversion rate isn't each month's transactions over its sessions.")
                    break
                if problems:
                    continue
                cells = book[year]
                letters = {cell.value: cell.column_letter for cell in cells[1]}
                formats = {"sessions": "#,##0", "revenue": "$#,##0.00", "conversion rate": "0.00%"}
                wrong = [column for column in columns if cells[letters[column] + "2"].number_format != next(number_format for metric, number_format in formats.items() if column.endswith(metric))]
                if wrong:
                    problems.append(f"Some columns on the {year} sheet don't have their number format, such as {wrong[0]} ({cells[letters[wrong[0]] + '2'].number_format}). Sessions are #,##0, revenue $#,##0.00 and conversion rates 0.00%.")
                elif any((cells.column_dimensions[column[0].column_letter].width or 0) < 16 for column in cells.columns):
                    problems.append(f"Some columns on the {year} sheet are narrower than 16.")
            if problems:
                results.append((False, problems[0]))
            else:
                email = back["2024"].loc["2024-11", "Email conversion rate"]
                total = back["2024"].loc["2024-11", "Total conversion rate"]
                results.append((True, f"{report_file} has both years, every number matches the store's data, and it's formatted: November 2024's Email converted {email:.2%}, and the store {total:.2%}."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. Download the file from the Files panel, and open it: this is the report.")


check_your_work()

<details>
<summary><b>Show hint</b></summary>

Paste `SOCIAL_SITES`, `clean()` and `channel_group()` from Foundations topic 12.3., and apply it with a row function to `session_source` and `session_medium`, not to the standardized `source` (topic 4.2.). The month is `.dt.strftime("%Y-%m")`.

For `monthly`, group by month and channel group with `as_index=False` and total the three columns (topic 4.3.). Group by month alone for the totals, give them the channel group `"Total"`, and stack the two with `pd.concat` (topic 2.5.). Calculate `conversion_rate` after that, from the totals of each row: never average a rate (topic 4.4.).

`report` is `monthly` pivoted with three `values` and its two-level column names flattened (topic 4.5.). `metric.replace("_", " ")` turns `conversion_rate` into `conversion rate`. For the file, loop over the two years, write each year's rows with `report.index.str.startswith(year)`, and format each column by how its name ends, with topic 4.6.'s `format_column()`.

</details>

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
SOCIAL_SITES = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}


def clean(value):
    """Return the value lowercased and stripped, or "" for None."""
    if value is None:
        return ""
    return value.strip().lower()


def channel_group(source, medium):
    """Return the channel group of a session's source and medium, with Foundations topic 12.3's rules."""
    source = clean(source)
    medium = clean(medium)
    if source == "(direct)" and medium == "(none)":
        return "Direct"
    if medium == "email":
        return "Email"
    if medium == "cpc":
        return "Paid Search"
    if medium == "display":
        return "Display"
    if medium == "paid_social":
        return "Paid Social"
    if medium == "video":
        return "Paid Video"
    if medium == "organic":
        return "Organic Search"
    if medium == "referral" and source in SOCIAL_SITES:
        return "Organic Social"
    if medium == "referral":
        return "Referral"
    return "Unassigned"


def row_channel_group(row):
    """Return the channel group of one row of the sessions table."""
    return channel_group(row["session_source"], row["session_medium"])


sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)
sessions["month"] = sessions["date"].dt.strftime("%Y-%m")
assert set(sessions["channel_group"].unique()) <= CUSTOMER_CHANNELS, set(sessions["channel_group"].unique()) - CUSTOMER_CHANNELS
sessions["channel_group"].value_counts()
```

```python
by_group = sessions.groupby(["month", "channel_group"], as_index=False)[["sessions", "transactions", "revenue"]].sum()
totals = sessions.groupby("month", as_index=False)[["sessions", "transactions", "revenue"]].sum()
totals["channel_group"] = "Total"
monthly = pd.concat([by_group, totals], ignore_index=True)
monthly["conversion_rate"] = monthly["transactions"] / monthly["sessions"]
assert monthly.loc[monthly["channel_group"] == "Total", "sessions"].sum() == sessions["sessions"].sum()

overall_rate = sessions["transactions"].sum() / sessions["sessions"].sum()
len(monthly), overall_rate
```

```python
report = monthly.pivot(index="month", columns="channel_group", values=["sessions", "revenue", "conversion_rate"])
report.columns = [f"{group} {metric.replace('_', ' ')}" for metric, group in report.columns]
report.shape
```

```python
def format_column(sheet, header, number_format):
    """Give every cell under the header the number format."""
    for cell in sheet[1]:
        if cell.value == header:
            for value_cell in sheet[cell.column_letter][1:]:
                value_cell.number_format = number_format


NUMBER_FORMATS = {"sessions": "#,##0", "revenue": "$#,##0.00", "conversion rate": "0.00%"}

with pd.ExcelWriter(REPORT_FILE, engine="openpyxl") as writer:
    for year in ["2024", "2025"]:
        report[report.index.str.startswith(year)].to_excel(writer, sheet_name=year)
        sheet = writer.sheets[year]
        for header in report.columns:
            for metric, number_format in NUMBER_FORMATS.items():
                if header.endswith(metric):
                    format_column(sheet, header, number_format)
        for column in sheet.columns:
            sheet.column_dimensions[column[0].column_letter].width = 16
```

```python
back = pd.read_excel(REPORT_FILE, sheet_name=None, index_col=0)
both = pd.concat([back["2024"], back["2025"]])
list(back), (both[report.columns] - report).abs().max().max()
```

</details>